# Infosys Quarterly Report Analysis [Multiple Documents]

In [ ]:
import nest_asyncio

nest_asyncio.apply()

In [ ]:
import os
from getpass import getpass
os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

In [ ]:
from llama_index.core import SimpleDirectoryReader, ServiceContext, VectorStoreIndex, StorageContext
from llama_index.core.response.pprint_utils import pprint_response
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.core.query_engine import SubQuestionQueryEngine
from llama_index.core.node_parser import SimpleNodeParser
from llama_index.core.node_parser import (SentenceWindowNodeParser,)
from llama_index.core.text_splitter import SentenceSplitter
from llama_index.core import Document
import faiss
from llama_index.vector_stores.faiss import FaissVectorStore
from llama_index.llms.groq import Groq
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

In [ ]:
import faiss

## Configure LLM service

In [ ]:
llm = Groq(model="llama-3.3-70b-versatile")

In [ ]:
embed_model = HuggingFaceEmbedding(model_name="Qwen/Qwen3-Embedding-0.6B")

In [ ]:
from llama_index.core import Settings

Settings.llm = llm
Settings.embed_model = embed_model
Settings.node_parser = SentenceSplitter(chunk_size=512, chunk_overlap=20)
Settings.num_output = 512
Settings.context_window = 4096

## Load data
Downloaded from

https://www.infosys.com/investors/reports-filings/quarterly-results.html

In [ ]:
!pip install docling

In [ ]:
from docling.document_converter import DocumentConverter

In [ ]:
def convert(file, ofile):
  converter = DocumentConverter()
  result = converter.convert(file)
  result.document.export_to_markdown()

  # Define the output directory
  output_dir = "output_documents"
  os.makedirs(output_dir, exist_ok=True)  # Ensure the directory exists

  # Define output file path
  output_path = os.path.join(output_dir, ofile)

  # Save as Markdown
  with open(output_path, "w", encoding="utf-8") as f:
      f.write(result.document.export_to_markdown())

  print(f"Document saved at: {output_path}")

In [ ]:
convert("./ifrs-inr-press-release_q1_2022.pdf", "./ifrs-inr-press-release_q1_2022.md")
convert("./ifrs-inr-press-release_q1_2023.pdf", "./ifrs-inr-press-release_q1_2023.md")
convert("./ifrs-inr-press-release_q1_2024.pdf", "./ifrs-inr-press-release_q1_2024.md")

In [ ]:
documents = [
    {
        "file_path": "./ifrs-inr-press-release_q1_2022.md",
        "metadata": {"year": "2022", "quarter": "Q1", "company": "Infosys"}
    },
    {
        "file_path": "./ifrs-inr-press-release_q1_2023.md",
        "metadata": {"year": "2023", "quarter": "Q1", "company": "Infosys"}
    },
    {
        "file_path": "./ifrs-inr-press-release_q1_2024.md",
        "metadata": {"year": "2024", "quarter": "Q1", "company": "Infosys"}
    }
]

In [ ]:
nodes = []
parser = SimpleNodeParser()

for doc in documents:
    reader = SimpleDirectoryReader(input_files=[doc["file_path"]])
    data = reader.load_data()

    # Attach metadata
    for d in data:
        d.metadata = doc["metadata"]

    nodes.extend(parser.get_nodes_from_documents(data, show_progress=True))

print(f"Loaded {len(nodes)} nodes from documents")

# Build indices

In [ ]:
# dimensions of text-ada-embedding-002
d = 1024
faiss_index = faiss.IndexFlatL2(d)

In [ ]:
vector_store = FaissVectorStore(faiss_index=faiss_index)
storage_context = StorageContext.from_defaults()
all_index = VectorStoreIndex(nodes, storage_context=storage_context)

## Build query engines

In [ ]:
q_engine = all_index.as_query_engine(similarity_top_k=5)

## Run queries

In [ ]:
response = q_engine.query(
    "What is the QoQ revenue growth in Q1 FY24?"
)

In [ ]:
pprint_response(response)

In [ ]:
response = q_engine.query("Can you compare the operating margins over First quarters in FY22, FY23 and FY24 in infosys")
pprint_response(response)

In [ ]:
response = q_engine.query("Can you compare the assets over First quarters in FY22, FY23 and FY24 in infosys")
pprint_response(response)